# 03 — Optional Offline Product Artifact

This notebook is separate from the main product URL discovery notebooks. Use it only after you already have a confirmed champion URL and want to freeze that live page into a local, openable offline artifact.

The main tournament flow remains unchanged:

```text
product discovery → champion confirmation → production URL gate
```

This optional notebook adds:

```text
confirmed champion URL → offline artifact capture → offline/offline_page.html
```


## What this creates

For one confirmed champion URL, this notebook creates:

```text
<artifact_dir>/
├── offline_artifact_manifest.json
├── live_capture/
│   ├── raw.html
│   ├── rendered.html
│   ├── rendered_clean.html
│   └── page_text.txt
├── product_data/
│   ├── content.md
│   └── structured_product.json
├── offline/
│   ├── offline_page.html
│   ├── asset_map.json
│   └── assets/
└── validation/
    └── offline_artifact_validation.json
```


In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_PATH = PROJECT_ROOT / 'src'

# Keep both paths available because the package currently contains internal src.* imports.
for path in (PROJECT_ROOT, SRC_PATH):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

PROJECT_ROOT


In [ ]:
from product_evidence_harness import ProductQuery
from product_evidence_harness.offline_capture import (
    LivePageOfflineArtifactBuilder,
    OfflineCaptureConfig,
)


## Input

Paste the confirmed champion URL here. This notebook should not be used for weak/review URLs.


In [ ]:
CHAMPION_URL = ''  # paste confirmed champion URL here

product = ProductQuery(
    row_id='demo-offline-001',
    main_text='PUT PRODUCT MAIN_TEXT HERE',
    country_code='CZ',
    retailer_name='',
    ean='',
)

assert CHAMPION_URL, 'Paste a confirmed champion URL before running capture.'


## Capture the page offline

This performs one live capture, downloads assets, rewrites references to local files, disables live scripts/forms/links, and writes validation metadata.


In [ ]:
builder = LivePageOfflineArtifactBuilder(
    OfflineCaptureConfig(
        output_dir=PROJECT_ROOT / 'outputs' / 'offline_artifacts',
        disable_scripts=True,
        disable_external_links=True,
        verify_no_network_bound_html=True,
    )
)

artifact = builder.capture_url(CHAMPION_URL, product=product)
artifact.to_dict()


## Review result

The offline page should be opened from `offline/offline_page.html`. The validation file tells whether it is safe as an offline artifact.


In [ ]:
print('status:', artifact.status)
print('offline_artifact_ready:', artifact.offline_artifact_ready)
print('offline html:', artifact.offline_html_path)
print('manifest:', artifact.manifest_path)
print('validation:', artifact.validation_path)


In [ ]:
validation = json.loads(Path(artifact.validation_path).read_text(encoding='utf-8'))
validation


## Handoff rule

Use the offline artifact downstream only when:

```text
offline_artifact_ready = true
status = PRODUCTION_READY_OFFLINE_ARTIFACT
validation.network_bound_reference_count = 0
```

This notebook is intentionally opt-in and separate from the main discovery notebooks.
